In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import spatial
import seaborn as sns
import copy
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# df = df.drop([0, 7]) #删除已知的两条脏数据
df = pd.read_excel(r"xxx")
y_name = '价格'

In [ ]:
df_describe=df[y_name].describe()
df_std=df_describe.loc['std']
df_std

In [ ]:
# 去除离群样本
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# 标准化数据
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df.loc[:, df.columns[1]:df.columns[-1]])

# 创建IsolationForest模型
clf = IsolationForest(contamination=0.05, random_state=42)
clf.fit(scaled_data)
#训练模型并预测异常值
df['is_outlier'] = clf.predict(scaled_data)
# df

In [ ]:
#忽略warning
import warnings
warnings.filterwarnings('ignore')
# 聚类，看看聚类标签与价格标签是否一致
# 找到最佳的k-轮廓系数法
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
silhouette_scores = []

k_values = range(2,10)
n_runs=10
for k in k_values:
    all_cluster_assignments = []
    for i in range(n_runs):
        kmeans = KMeans(n_clusters = k, random_state=42)
        kmeans.fit(scaled_data)
        all_cluster_assignments.append(kmeans)
    
    # 通过比较不同运行的簇内平方和来选择最优结果
    best_cluster_assignment = min(all_cluster_assignments, key = lambda x: x.inertia_)
    silhouette_scores.append(silhouette_score(scaled_data, best_cluster_assignment.labels_))
    
# 绘制轮廓系数法图形
plt.plot(k_values, silhouette_scores, marker = 'o')
plt.xlabel('簇的数量(k)')
plt.ylabel('轮廓系数')
plt.title('簇的轮廓系数')
plt.show()

In [ ]:
# 聚类，看看聚类标签与价格标签是否一致
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 标准化数据
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df.loc[:,df.columns[1]:df.columns[-1]])
# 设置聚类的簇数
num_clusters = 4

# 运行10次KMeans，比较簇内平方和，选择最佳模型
n_runs = 10
best_kmeans = None
lowest_inertia = float('inf')  # 初始化最小簇内平方和
best_cluster_assignments = None

for i in range(n_runs):
    kmeans = KMeans(n_clusters=num_clusters, random_state=i)
    cluster_assignments = kmeans.fit_predict(scaled_data)
    
    if kmeans.inertia_ < lowest_inertia:  # 比较簇内平方和
        lowest_inertia = kmeans.inertia_
        best_kmeans = kmeans  # 保存当前最佳模型
        best_cluster_assignments = cluster_assignments

# # 使用K_means聚类，运行10次
# n_runs=10
# all_cluster_assignments = []
# for i in range(n_runs):
#     kmeans = KMeans(n_clusters = num_clusters, random_state=i)
#     cluster_assignments = kmeans.fit_predict(scaled_data)
#     all_cluster_assignments.append(cluster_assignments)
    
# # 通过比较不同运行的簇内平方和来选择最优结果
# best_cluster_assignment = min(all_cluster_assignments, key = lambda x: kmeans.inertia_)
df['Cluster'] = best_cluster_assignments
# df

In [ ]:
# 设置价格区间
# df['价格类别'] = pd.cut(df['价格'], bins = [float('-inf'),35000,70000,105000,140000,175000,210000,245000,float('inf')],labels=[0,1,2,3,4,5,6,7])
# df['价格类别'] = pd.cut(df['价格'], bins = [float('-inf'),188103,245606,326251,float('inf')],labels=[0,1,2,3])
# df.to_excel('price_class_cluster_label_data1.xlsx', index=True)
df['价格类别'] = pd.qcut(df['价格'], q=num_clusters, labels=range(num_clusters))
price_bins = pd.qcut(df['价格'], q=num_clusters, retbins=True)
print(price_bins[1])  # 输出生成的边界区间

# 将结果保存到 Excel
df.to_excel('price_class_cluster_label_data.xlsx', index=False)

print(df[['价格', '价格类别']].head())

In [ ]:
df = df[(df['is_outlier'] != -1) ]

In [ ]:
cross_tab = pd.crosstab(df['价格类别'],df['Cluster'])

In [ ]:
import seaborn as sns
heatmap_data = cross_tab

# 绘制热力图
sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Correlation Heatmap: 价格类别 vs. cluster')

In [ ]:
from sklearn.model_selection import train_test_split
import random
# 生成一个随机数种子并保存到变量中
# randomnumber = random.randint(1, 10000)
numbers = [2491,9812,6613,5415,2045,1002,4692,4711,1008,6803,8261,347,7471,1917,2527,4753,4860,8411,7657,8388,5657,5638]
randomnumber = random.choice(numbers)
print("随机数种子：",randomnumber)
# 价格分层抽样
df = df.drop(columns=['is_outlier'])
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['价格类别'], random_state=randomnumber)
# 删除”价格类别"列
train_df = train_df.drop(columns=['价格类别'])
test_df = test_df.drop(columns=['价格类别'])
# train_df, test_df

In [ ]:
df_X = df.loc[:,df.columns[1]:'Cluster']
df_y = df[y_name]
X_train = train_df.loc[:,df.columns[1]:'Cluster']
y_train = train_df[y_name]
y_train = np.log(y_train)
X_test = test_df.loc[:, df.columns[1]:'Cluster']
y_test = test_df[y_name]
# 打印训练数据集和测试数据集的观测数量
print('训练数据集共有%d条观测' % X_train.shape[0])
print('测试数据集共有%d条观测' % X_test.shape[0])
# y_train
# y_test

In [ ]:
# 数据标准化
from sklearn.preprocessing import MinMaxScaler  
scaler = MinMaxScaler(feature_range=(0, 1))  # 指定缩放的范围  
scaler.fit(X_train)  # 用训练集的数据来拟合scaler  
# 用训练集的数据来拟合scaler并进行标准化
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # 注意这里使用transform而不是apply

# 保存原始的X_test
X_test_original = X_test.copy()

# 将标准化后的数组转换回DataFrame
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)


# X_train

X_test.to_excel('X_test.xlsx')
y_test.to_excel('y_test.xlsx')



In [ ]:
# In[15]:

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_predict
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

# # 创建线性回归模型
# linear_model = LinearRegression()
# # 通过交叉验证选择最佳模型
# predict = cross_val_predict(linear_model, X_train, y_train, cv=10)  # 使用 10 折交叉验证

# # 在训练集上训练模型
# linear_model.fit(X_train, y_train)
# # 在测试集上进行预测
# linear_predictions = linear_model.predict(X_test)
# max_allowed_log_value = np.log(np.finfo(np.float64).max)
# linear_predictions = np.clip(linear_predictions,a_min=None,a_max=max_allowed_log_value)
# linear_predictions = np.exp(linear_predictions)

# 模型分析
def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred) / y_true) * 100
# # 计算平均绝对百分比误差
# model1_mape = mean_absolute_percentage_error(y_test, linear_predictions)

# print("LR模型的平均绝对百分比误差：", model1_mape)
# # 计算每个样本的残差
# residuals = np.exp(y_train)-np.exp(predict)
# #计算残差的标准差作为置信度估计
# confidence = np.std(residuals)
# model1_mse = mean_squared_error(y_test,  linear_predictions)
# print("均方误差 (MSE):", model1_mse)
# model1_rmse = np.sqrt(model1_mse)
# print("均方根误差 (RMSE):", model1_rmse)
# model1_mae = mean_absolute_error(y_test,  linear_predictions)
# print("平均绝对误差 (MAE):", model1_mae)
# model1_r2 = r2_score(y_test,  linear_predictions)
# print("R^2 得分:", model1_r2)
# print("预测值的置信度（标准差）：",confidence)

In [ ]:
def draw_result(predictions, X_test, y_test):
    plt.rcParams['font.sans-serif'] = ['SimHei']
    plt.rcParams['axes.unicode_minus'] = False
    X = range(1, len(X_test) + 1)
    bar_width = 0.4  # 柱状图宽度

    plt.figure(figsize=(16, 9))
    error = np.abs(np.array(y_test) - np.array(predictions)) / 2
    # 将折线图改为分组柱状图
    plt.bar(X, y_test, width=bar_width, label='真实值', color='orange', alpha=0.5)
    plt.bar([x + bar_width for x in X], predictions, width=bar_width, label='预测值', color='b', alpha=0.5,
           yerr = error, capsize=5)
    mapes =  np.abs(y_test - predictions) / y_test
    # 添加误差值标签
    for x, y, mape in zip(X, predictions, mapes):
        plt.text(x + bar_width, y + mape + 0.01, '{:.2%}'.format(mape), ha='left',  fontsize=12 )


    plt.ylabel('价格', fontsize=20)
    plt.xlabel('测试集', fontsize=20)
    plt.tick_params(labelsize=20)
    plt.legend(['真实值', '预测值'], fontsize=30)
    plt.title('在测试集上的预测效果', fontsize=30)
    plt.xticks([x + bar_width / 2 for x in X], X)
    plt.savefig('result.png')
    plt.show()
    #把图片保存到本地

In [ ]:
from matplotlib.ticker import FuncFormatter
def draw_mapes_hist(y_test, predictions):
    mapes = np.abs(y_test - predictions) / y_test

    plt.hist(mapes, bins = 5, color = 'blue', alpha =0.7)
    def percent_format(x, pos):
        return f'{x * 100:.2f}%'

    plt.gca().xaxis.set_major_formatter(FuncFormatter(percent_format))
    plt.xlabel('值')
    plt.ylabel('频数')
    plt.title('mape直方图')
    plt.show()

In [ ]:
from sklearn.utils import resample
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score

# 计算置信区间 直接计算法
def calculate_confidence_intervals(predicted_values, std_devs):
    z_value = 1 

    lower_bounds = predicted_values - z_value * std_devs
    upper_bounds = predicted_values + z_value * std_devs

    return lower_bounds, upper_bounds


#新的三条标准
def new_3_indictors(df_X, df_y, X_test, y_test, pred, model, model_name):
 
    # 1. 计算 R² 分数
    r2 = r2_score(y_test, pred)
    print(f'R² score: {r2}')

    
    
    residuals = y_test-pred
#计算残差的标准差作为置信度估计
#     confidence = np.std(residuals)
    confidence = np.std(pred)

    lower_bounds, upper_bounds = calculate_confidence_intervals(pred, confidence)

    ci_width = upper_bounds - lower_bounds

    print(f'Confidence Interval Width: {ci_width}')

    
    
    # 3. 计算模型稳定性（使用平均绝对误差百分比）
    n_splits = min(5, len(X_test))  # 确保 n_splits 小于等于样本数量
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=1) # 交叉验证
    mape_scores = []

    for train_index, test_index in cv.split(df_X):
        X_cv_train, X_cv_test = df_X.iloc[train_index], df_X.iloc[test_index]
        y_cv_train, y_cv_test = df_y.iloc[train_index], df_y.iloc[test_index]
        model.fit(X_cv_train, y_cv_train)  # 重新训练模型
        y_cv_pred = model.predict(X_cv_test)
        mape = mean_absolute_percentage_error(y_cv_test, y_cv_pred)
        mape_scores.append(mape)

    # 计算MAPE的标准差和平均值
    mape_scores = np.array(mape_scores)
    stability = np.std(mape_scores)
    print(f'Model Stability (Std of MAPE): {stability}')

    # 输出评估结果
    if r2 > 0.6:
        print("R² score is acceptable.")
    else:
        print("R² score is too low.")

    specific_width_value = df_std*2  # bootstap置信区间宽度阈值
    if np.mean(ci_width) < specific_width_value:
        print("Confidence interval width is acceptable.")
    else:
        print("Confidence interval width is too wide.")
        
    specific_stability_value = 20  # 稳定性阈值百分比
    if stability < specific_stability_value:
        print("Model stability is acceptable.")
    else:
        print("Model stability is too low.")
        
    return r2, ci_width, stability

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error

def lasso_regression(X_train, y_train, X_test, y_test):
    # 创建候选 alpha 列表
    alphas = [0.00001,0.0001,0.001, 0.01, 0.1, 1, 10]

    best_alpha = None
    best_score = float('inf')  # 修改为初始值为正无穷大

    # 通过交叉验证选择最佳 alpha
    for alpha in alphas:
        lasso_model = Lasso(alpha=alpha, max_iter=10000)  # 增加 max_iter 参数
        lasso_model.fit(X_train, y_train)  # 在训练集上训练模型
        lasso_predictions = lasso_model.predict(X_test)  # 在测试集上进行预测
#         max_allowed_log_value = np.log(np.finfo(np.float64).max)
#         lasso_predictions = np.clip(lasso_predictions,a_min=None,a_max=max_allowed_log_value)
        lasso_predictions = np.exp(lasso_predictions)
        mae = mean_absolute_error(y_test, lasso_predictions)  # 计算均方绝对误差

        if mae < best_score:  # 修改判断条件，选取最小的均方绝对误差
            best_score = mae
            best_alpha = alpha

    print("Lasso模型的最佳正则化参数：", best_alpha)

    # 创建Lasso回归模型
    lasso_model = Lasso(alpha=best_alpha, max_iter=10000)  # 使用最佳的 alpha 和增加的 max_iter

    # 在训练集上训练模型
    lasso_model.fit(X_train, y_train)

    # 在测试集上进行预测
    lasso_predictions = lasso_model.predict(X_test)
#     max_allowed_log_value = np.log(np.finfo(np.float64).max)
#     lasso_predictions = np.clip(lasso_predictions,a_min=None,a_max=max_allowed_log_value)
    lasso_predictions = np.exp(lasso_predictions)

    # 模型分析
    # 计算每个测试样本的残差
    residuals = y_test - lasso_predictions

    # 计算残差的标准差
    std_dev = np.std(residuals)

    model2_mse = mean_squared_error(y_test,  lasso_predictions)
    print("均方误差 (MSE):", model2_mse)
    model2_rmse = np.sqrt(model2_mse)
    print("均方根误差 (RMSE):", model2_rmse)
    model2_mae = mean_absolute_error(y_test,  lasso_predictions)
    print("平均绝对误差 (MAE):", model2_mae)
    model2_r2 = r2_score(y_test,  lasso_predictions)
    print("R^2 得分:", model2_r2)
    lasso_mape = mean_absolute_percentage_error(y_test, lasso_predictions)
    print("Lasso模型的平均绝对百分比误差：{:.2f}%".format(lasso_mape))
    lower_bounds, upper_bounds = calculate_confidence_intervals(lasso_predictions, std_dev)
    confidence_intervals = list(zip(lower_bounds, upper_bounds))
    print("95% 的置信区间列表:", confidence_intervals)
    return lasso_model, lasso_predictions
lasso_model, lasso_predictions = lasso_regression(X_train, y_train, X_test, y_test)

In [ ]:
lasso_r2, lasso_ci_width, lasso_stability = new_3_indictors(df_X, df_y,X_test, y_test, lasso_predictions, lasso_model, 'Lasso')

In [ ]:
draw_result(lasso_predictions, X_test, y_test)
draw_mapes_hist(y_test, lasso_predictions)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge  # 使用岭回归
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def ridge_regression(X_train, y_train, X_test, y_test):
    # 创建候选 alpha 列表
    alphas = [0.00001,0.0001,0.001, 0.01, 0.1, 1, 10,100]

    best_alpha = None
    best_score = float('inf')  # 修改为初始值为正无穷大

    # 通过交叉验证选择最佳 alpha
    for alpha in alphas:
        ridge_model = Ridge(alpha=alpha)  # 使用岭回归
        ridge_model.fit(X_train, y_train)  # 在训练集上训练模型
        ridge_predictions = ridge_model.predict(X_test)  # 在测试集上进行预测
        #     max_allowed_log_value = np.log(np.finfo(np.float64).max)
#      lasso_predictions = np.clip(lasso_predictions,a_min=None,a_max=max_allowed_log_value)
        ridge_predictions = np.exp(ridge_predictions)
        mse = mean_squared_error(y_test, ridge_predictions)  # 计算均方误差

        if mse < best_score:  # 修改判断条件，选取最小的均方误差
            best_score = mse
            best_alpha = alpha

    print("岭回归模型的最佳正则化参数：", best_alpha)

    # 创建岭回归模型
    ridge_model = Ridge(alpha=best_alpha)  # 使用最佳的 alpha

    # 在训练集上训练模型
    ridge_model.fit(X_train, y_train)

    # 在测试集上进行预测
    ridge_predictions = ridge_model.predict(X_test)
    #     max_allowed_log_value = np.log(np.finfo(np.float64).max)
#     lasso_predictions = np.clip(lasso_predictions,a_min=None,a_max=max_allowed_log_value)
    ridge_predictions = np.exp(ridge_predictions)

    # 模型分析
    # 计算每个测试样本的残差
    residuals = y_test - ridge_predictions

    # 计算残差的标准差
    std_dev = np.std(residuals)
    model3_mse = mean_squared_error(y_test, ridge_predictions)
    print("均方误差 (MSE):", model3_mse)
    model3_rmse = np.sqrt(model3_mse)
    print("均方根误差 (RMSE):", model3_rmse)
    model3_mae = mean_absolute_error(y_test, ridge_predictions)
    print("平均绝对误差 (MAE):", model3_mae)
    model3_r2 = r2_score(y_test, ridge_predictions)
    print("R^2 得分:", model3_r2)
    lower_bounds, upper_bounds = calculate_confidence_intervals(ridge_predictions, std_dev)
    confidence_intervals = list(zip(lower_bounds, upper_bounds))
    print("95% 的置信区间列表:", confidence_intervals)
    RGlibi=abs(ridge_predictions/y_test-1)
    print("岭回归模型的平均绝对误差百分比:",RGlibi.mean())
    return ridge_model, ridge_predictions
ridge_model,ridge_predictions = ridge_regression(X_train, y_train, X_test, y_test)

In [ ]:
ridge_r2, ridge_ci_width, ridge_stability = new_3_indictors(df_X, df_y,X_test, y_test, ridge_predictions, ridge_model, 'Ridge')

In [ ]:
draw_result(ridge_predictions, X_test, y_test)
draw_mapes_hist(y_test, ridge_predictions)

In [ ]:
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
def svr(X_train, y_train, X_test, y_test):
    
        
    # 创建SVR模型
    svr = SVR()
    param_grid = {
        'kernel':['linear','poly','rbf','sigmoid'],#核函数
        'C':[0.001,0.01,0.1,1,10,100,1000,10000,100000],
        'epsilon':[0.01,0.1,0.2],
        'gamma':[0.001,0.01,0.1,1]
    }

    # 使用GridSearchCV进行参数搜索和交叉验证
    grid_svr = GridSearchCV(svr, param_grid=param_grid, cv=10, scoring='neg_mean_absolute_error')
    grid_svr.fit(X_train, y_train)
    # 输出测试集上的预测结果
    svr_model = grid_svr.best_estimator_   
    
    print("svr最好参数组合：", grid_svr.best_params_)

    grid_svr_pred = svr_model.predict(X_test)
    grid_svr_pred = np.exp(grid_svr_pred)
    
    # 模型分析
    # 计算每个测试样本的残差
    residuals = y_test - grid_svr_pred

    # 计算残差的标准差
    std_dev = np.std(residuals)
    model4_mse = mean_squared_error(y_test,grid_svr_pred)
    print("均方误差 (MSE):", model4_mse)
    model4_rmse = np.sqrt(model4_mse)
    print("均方根误差 (RMSE):", model4_rmse)
    model4_mae = mean_absolute_error(y_test, grid_svr_pred)
    print("平均绝对误差 (MAE):", model4_mae)
    model4_r2 = r2_score(y_test, grid_svr_pred)
    print("R^2 得分:", model4_r2)
    lower_bounds, upper_bounds = calculate_confidence_intervals(grid_svr_pred, std_dev)
    confidence_intervals = list(zip(lower_bounds, upper_bounds))
    print("95% 的置信区间列表:", confidence_intervals)
    SVRlibi=abs(grid_svr_pred /y_test-1)
    print("平均绝对误差百分比:",SVRlibi.mean())
    return svr_model, grid_svr_pred
svr_model, grid_svr_pred = svr(X_train, y_train, X_test, y_test)
draw_result(grid_svr_pred, X_test, y_test)

In [ ]:
svr_r2, svr_ci_width, svr_stability = new_3_indictors(df_X, df_y,X_test, y_test, grid_svr_pred, svr_model, 'SVR')

In [ ]:
draw_mapes_hist(y_test, grid_svr_pred)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus']=False
def gbdt(X_train, y_train, X_test, y_test):
    # 定义多组参数组合，包括 loss 参数
    param_grid = {
        'min_samples_split': [2,4,6,8,10],   # 内部节点最小样本数
        'min_samples_leaf': [1,3,5,7,9],     # 叶子节点最小样本数
        'loss': ['huber'],  # 损失函数选项
        'max_depth': [3,5,7,9],
        'n_estimators': [50,100,300,500],
        'learning_rate': [0.01,0.1,0.3],
        'subsample':[0.5,0.7,0.9,1]
    }

    # 创建GradientBoostingRegressor模型
    gbdt = GradientBoostingRegressor()

    # 使用GridSearchCV进行参数搜索和交叉验证
    grid_gbdt = GridSearchCV(estimator=gbdt, param_grid=param_grid, cv=10, scoring='neg_mean_absolute_error',n_jobs=-1)
    grid_gbdt.fit(X_train, y_train)
    # 输出测试集上的预测结果
    gbdt_model = grid_gbdt.best_estimator_   
    print("最好参数",grid_gbdt.best_params_)

    grid_gbdt_pred = gbdt_model.predict(X_test)
    grid_gbdt_pred = np.exp(grid_gbdt_pred)

    # 模型分析

    # 计算每个测试样本的残差
    residuals = y_test - grid_gbdt_pred

    # 计算残差的标准差
    std_dev = np.std(residuals)

    model5_mse = mean_squared_error(y_test, grid_gbdt_pred)
    print("均方误差 (MSE):", model5_mse)
    model5_rmse = np.sqrt(model5_mse)
    print("均方根误差 (RMSE):", model5_rmse)
    model5_mae = mean_absolute_error(y_test, grid_gbdt_pred)
    print("平均绝对误差 (MAE):", model5_mae)
    model5_r2 = r2_score(y_test, grid_gbdt_pred)
    print("R^2 得分:", model5_r2)
    lower_bounds, upper_bounds = calculate_confidence_intervals(grid_gbdt_pred, std_dev)
    confidence_intervals = list(zip(lower_bounds, upper_bounds))
    print("95% 的置信区间列表:", confidence_intervals)
    GBDTbili=abs(grid_gbdt_pred/y_test-1)
    print("平均绝对误差百分比:",GBDTbili.mean())
    return gbdt_model, grid_gbdt_pred
gbdt_model, grid_gbdt_pred = gbdt(X_train, y_train, X_test, y_test)
draw_result(grid_gbdt_pred, X_test, y_test)

In [ ]:
gbdt_r2, gbdt_ci_width, gbdt_stability = new_3_indictors(df_X, df_y,X_test, y_test, grid_gbdt_pred, gbdt_model, 'GBDT')

In [ ]:
gbdt_importance = gbdt_model.feature_importances_
plt.figure(figsize=(10,20))
plt.barh(range(len(gbdt_importance)),gbdt_importance,tick_label=X_train.columns)
plt.xlabel('Feature importance')
plt.title('gbdt feature importance')
plt.show()
print(gbdt_importance)

In [ ]:
draw_mapes_hist(y_test, grid_gbdt_pred)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import ExtraTreesRegressor
import matplotlib.pyplot as plt
def et(X_train, y_train, X_test, y_test):
    # 超参数搜索范围
    max_depth_options = [10, 20, 30]  # 不同的最大深度值
    n_estimators_options = [50, 100, 300, 500]  # 不同的树的数量
    min_samples_split = [2, 4, 6, 8, 10]  # 不同的最小样本拆分值
    min_samples_leaf = [1, 3, 5, 7, 9]

    parameters = {
        'max_depth': max_depth_options,
        'n_estimators': n_estimators_options,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf
    }

    # 创建GridSearchCV对象
    grid_et = GridSearchCV(estimator=ExtraTreesRegressor(), param_grid=parameters, cv=10, scoring='neg_mean_absolute_error', n_jobs=-1)
    grid_et.fit(X_train, y_train)
    # 获取最佳参数组合
    best_params = grid_et.best_params_
    # 创建 ExtraTreesRegressor 模型
    et_model = ExtraTreesRegressor(max_depth=best_params['max_depth'], 
                                   n_estimators=best_params['n_estimators'], 
                                   min_samples_split=best_params['min_samples_split'],
                                   min_samples_leaf=best_params['min_samples_leaf']
                                  )

    # 拟合 ExtraTreesRegressor 模型
    et_model.fit(X_train, y_train)
    print("最好",best_params)
    # 使用模型进行预测
    grid_et_pred = et_model.predict(X_test)
    grid_et_pred = np.exp(grid_et_pred)

    # 模型分析
    # 计算每个测试样本的残差
    residuals = y_test - grid_et_pred

    # 计算残差的标准差
    std_dev = np.std(residuals)
    model6_mse = mean_squared_error(y_test, grid_et_pred)
    print("均方误差 (MSE):", model6_mse)
    model6_rmse = np.sqrt(model6_mse)
    print("均方根误差 (RMSE):", model6_rmse)
    model6_mae = mean_absolute_error(y_test, grid_et_pred)
    print("平均绝对误差 (MAE):", model6_mae)
    model6_r2 = r2_score(y_test, grid_et_pred)
    print("R^2 得分:", model6_r2)
    lower_bounds, upper_bounds = calculate_confidence_intervals(grid_et_pred, std_dev)
    confidence_intervals = list(zip(lower_bounds, upper_bounds))
    print("95% 的置信区间列表:", confidence_intervals)
    ETlibi=abs(grid_et_pred/y_test-1)
    print("平均绝对误差百分比:",ETlibi.mean())
    return et_model, grid_et_pred
et_model, grid_et_pred = et(X_train, y_train, X_test, y_test)
draw_result(grid_et_pred, X_test, y_test)
draw_mapes_hist(y_test, grid_et_pred)

In [ ]:
et_r2, et_ci_width, et_stability = new_3_indictors(df_X, df_y,X_test, y_test, grid_et_pred, et_model, 'ET')

In [ ]:
et_importance = et_model.feature_importances_
plt.figure(figsize=(10,20))
plt.barh(range(len(et_importance)),et_importance,tick_label=X_train.columns)
plt.xlabel('Feature importance')
plt.title('et feature importance')
plt.show()
print(et_importance)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt

def rf(X_train, y_train, X_test, y_test):
    # 超参数搜索范围
    max_depth_options = [10, 20, 30]  # 不同的最大深度值
    n_estimators_options = [50, 100, 300, 500]  # 不同的树的数量
    min_samples_split = [2, 4, 6, 8, 10]  # 不同的最小样本拆分值
    min_samples_leaf = [1, 3, 5, 7, 9]

    parameters = {
        'max_depth': max_depth_options,
        'n_estimators': n_estimators_options,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf
    }

    # 创建 GridSearchCV 对象
    grid_rf = GridSearchCV(
        estimator=RandomForestRegressor(),
        param_grid=parameters,
        cv=10,  # 交叉验证折数
        scoring='neg_mean_absolute_error',  # 使用负平均绝对误差作为回归问题的评估指标
        n_jobs=6
    )

    grid_rf.fit(X_train, y_train)

    # 创建 RandomForestRegressor 模型
    rf_model = RandomForestRegressor(max_depth=grid_rf.best_params_['max_depth'],
                                     n_estimators=grid_rf.best_params_['n_estimators'],
                                     min_samples_split=grid_rf.best_params_['min_samples_split'],
                                     min_samples_leaf=grid_rf.best_params_['min_samples_leaf']
                                    )

    # 拟合 RandomForestRegressor 模型
    rf_model.fit(X_train, y_train)
    print("最好",grid_rf.best_params_)
    # 使用模型进行预测
    grid_rf_pred = rf_model.predict(X_test)
    grid_rf_pred = np.exp(grid_rf_pred)
    # 模型分析
    # 计算每个测试样本的残差
    residuals = y_test - grid_rf_pred

    # 计算残差的标准差
    std_dev = np.std(residuals)
    model7_mse = mean_squared_error(y_test, grid_rf_pred)
    print("均方误差 (MSE):", model7_mse)
    model7_rmse = np.sqrt(model7_mse)
    print("均方根误差 (RMSE):", model7_rmse)
    model7_mae = mean_absolute_error(y_test, grid_rf_pred)
    print("平均绝对误差 (MAE):", model7_mae)
    model7_r2 = r2_score(y_test, grid_rf_pred)
    print("R^2 得分:", model7_r2)
    lower_bounds, upper_bounds = calculate_confidence_intervals(grid_rf_pred, std_dev)
    confidence_intervals = list(zip(lower_bounds, upper_bounds))
    print("95% 的置信区间列表:", confidence_intervals)
    RFlibi=abs(grid_rf_pred/y_test-1)
    print("平均绝对误差百分比:",RFlibi.mean())
    return rf_model, grid_rf_pred
rf_model, grid_rf_pred = rf(X_train, y_train, X_test, y_test)
draw_result(grid_rf_pred, X_test, y_test)
draw_mapes_hist(y_test, grid_rf_pred)

In [ ]:
rf_r2, rf_ci_width, rf_stability = new_3_indictors(df_X, df_y,X_test, y_test, grid_rf_pred, rf_model, 'RF')

In [ ]:
rf_importance = rf_model.feature_importances_
plt.figure(figsize=(10,20))
plt.barh(range(len(rf_importance)),rf_importance,tick_label=X_train.columns)
plt.xlabel('Feature importance')
plt.title('rf feature importance')
plt.show()
# print(rf_importance)

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error

def xgboost(X_train, y_train, X_test, y_test):
    parameters = {
        'max_depth': [3,5,7,9],
        'n_estimators': [50,100,300,500],
        'learning_rate': [0.01,0.1,0.3],
        'min_child_weight': [1, 3, 5,7,9],
        'subsample':[0.5,0.7,0.9,1],
        'colsample_bytree':[0.5,0.7,0.9,1]
    }

    # 创建 GridSearchCV 对象
    grid_xgb = GridSearchCV(
        estimator=xgb.XGBRegressor(),
        param_grid=parameters,
        cv=10,  # 交叉验证折数
        scoring='neg_mean_absolute_error',  # 使用负平均绝对误差作为回归问题的评估指标
        n_jobs=6
    )

    grid_xgb.fit(X_train, y_train)

    # 创建 XGBoostRegressor 模型
    xgb_model = xgb.XGBRegressor(max_depth=grid_xgb.best_params_['max_depth'],
                                 n_estimators=grid_xgb.best_params_['n_estimators'],
                                 min_child_weight=grid_xgb.best_params_['min_child_weight'],
                                 learning_rate=grid_xgb.best_params_['learning_rate'],
                                 subsample=grid_xgb.best_params_['subsample'],
                                 colsample_bytree=grid_xgb.best_params_['colsample_bytree'],
                                 )

    # 训练模型
    xgb_model.fit(X_train, y_train)
    print("最好",grid_xgb.best_params_)
    # 在测试集上进行预测
    grid_xgb_pred = xgb_model.predict(X_test)
    grid_xgb_pred = np.exp(grid_xgb_pred)
    # 模型分析
    # 计算每个测试样本的残差
    residuals = y_test - grid_xgb_pred

    # 计算残差的标准差
    std_dev = np.std(residuals)

    model8_mse = mean_squared_error(y_test, grid_xgb_pred)
    print("均方误差 (MSE):", model8_mse)
    model8_rmse = np.sqrt(model8_mse)
    print("均方根误差 (RMSE):", model8_rmse)
    model8_mae = mean_absolute_error(y_test, grid_xgb_pred)
    print("平均绝对误差 (MAE):", model8_mae)
    model8_r2 = r2_score(y_test, grid_xgb_pred)
    print("R^2 得分:", model8_r2)
    lower_bounds, upper_bounds = calculate_confidence_intervals(grid_xgb_pred, std_dev)
    confidence_intervals = list(zip(lower_bounds, upper_bounds))
    print("95% 的置信区间列表:", confidence_intervals)
    XGlibi=abs(grid_xgb_pred/y_test-1)
    print("平均绝对误差百分比:",XGlibi.mean())
    return xgb_model, grid_xgb_pred
xgb_model, grid_xgb_pred = xgboost(X_train, y_train, X_test, y_test)
draw_result(grid_xgb_pred, X_test, y_test)
draw_mapes_hist(y_test, grid_xgb_pred)

In [ ]:
xgb_r2, xgb_ci_width, xgb_stability = new_3_indictors(df_X, df_y,X_test, y_test, grid_xgb_pred, xgb_model, 'XGBoost')

In [ ]:
xgb_importance = xgb_model.feature_importances_
plt.figure(figsize=(10,20))
plt.barh(range(len(xgb_importance)),xgb_importance,tick_label=X_train.columns)
plt.xlabel('Feature importance')
plt.title('xgb feature importance')
plt.show()
print(xgb_importance)

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest

# 创建一个包含所有数据的数组，其中每一行是一种模型的预测结果
data = np.array([
    lasso_predictions,
    ridge_predictions,
    grid_xgb_pred,
    grid_rf_pred,
    grid_et_pred,
    grid_svr_pred,
    grid_gbdt_pred
])

lasso_mape_weight = mean_absolute_percentage_error(y_test, lasso_predictions)
ridge_mape_weight = mean_absolute_percentage_error(y_test, ridge_predictions)
xgb_mape_weight = mean_absolute_percentage_error(y_test, grid_xgb_pred)
rf_mape_weight = mean_absolute_percentage_error(y_test, grid_rf_pred)
et_mape_weight = mean_absolute_percentage_error(y_test, grid_et_pred)
svr_mape_weight = mean_absolute_percentage_error(y_test, grid_svr_pred)
gbdt_mape_weight = mean_absolute_percentage_error(y_test, grid_gbdt_pred)

weight_score = 99-max(lasso_mape_weight,
                   ridge_mape_weight,
                   xgb_mape_weight,
                   rf_mape_weight,
                   et_mape_weight,
                   svr_mape_weight,
                   gbdt_mape_weight)

lasso_weight = 100 - mean_absolute_percentage_error(y_test, lasso_predictions)-weight_score
ridge_weight = 100 - mean_absolute_percentage_error(y_test, ridge_predictions)-weight_score
xgb_weight = 100 - mean_absolute_percentage_error(y_test, grid_xgb_pred)-weight_score
rf_weight = 100 - mean_absolute_percentage_error(y_test, grid_rf_pred)-weight_score
et_weight = 100 - mean_absolute_percentage_error(y_test, grid_et_pred)-weight_score
svr_weight = 100 - mean_absolute_percentage_error(y_test, grid_svr_pred)-weight_score
gbdt_weight = 100 - mean_absolute_percentage_error(y_test, grid_gbdt_pred)-weight_score
weights = np.array([lasso_weight,
                    ridge_weight,
                    xgb_weight,
                    rf_weight,
                    et_weight,
                    svr_weight,
                    gbdt_weight])
print("weight：", weights)
weights /= np.sum(weights)

print("weight：", weights)

print("原y_pred：")
print(data.T)
# 初始化孤立森林模型
clf = IsolationForest(n_estimators=300, 
                      max_samples='auto', 
                      contamination=float(0.1),
                      max_features=1.0)  # contamination参数表示预期的离群值比例

# 存储每列的平均值
column_means = []
cleaned_data = []
ocolumn_means = []

# 对每列数据进行拟合和预测
for column in data.T:  # 转置以便迭代列
    # 拟合孤立森林模型
    clf.fit(column.reshape(-1, 1))  # 转换为列向量
    
    # 预测离群值
    predictions = clf.predict(column.reshape(-1, 1))
    
    # 去除离群值并计算平均值
#     clean_column = column[predictions == 1]  # 去除离群值
#     cleaned_data.append(clean_column)
    clean_column = column.copy()
    clean_column[predictions == -1] = np.nan  # 替换离群值为NaN
    cleaned_data.append(clean_column)
    column_mean = np.nanmean(clean_column)   # 计算平均值
    column_means.append(column_mean)

# column_means 现在包含了每列数据的平均值
cleaned_data = np.array(cleaned_data)
print("去除离群值后的y_pred:")
print(cleaned_data)
print("每列数据的平均值:", column_means)

weight_column_means = np.dot(data.T, weights)
print("权重每列数据的平均值:", weight_column_means)

weight_mean_absolute_error_percentage = np.mean(np.abs((y_test - weight_column_means) / y_test) )
print("权重平均绝对误差百分比:", weight_mean_absolute_error_percentage)
draw_result(weight_column_means, X_test, y_test)

for column in data.T:  # 转置以便迭代列
    o_column = column.copy()
    ocolumn_mean = np.mean(o_column)
    ocolumn_means.append(ocolumn_mean)

print("没去除坏值的每列数据的平均值:", ocolumn_means)

# 计算没去除坏值的平均绝对误差百分比
no_drop_mean_absolute_error_percentage = np.mean(np.abs((y_test - ocolumn_means) / y_test) )
print("没去除坏值的平均绝对误差百分比:", no_drop_mean_absolute_error_percentage)
draw_result(ocolumn_means, X_test, y_test)

# 计算去除坏值的平均绝对误差百分比
drop_mean_absolute_error_percentage = np.mean(np.abs((y_test - column_means) / y_test) )
print("平均绝对误差百分比:", drop_mean_absolute_error_percentage)
draw_result(column_means, X_test, y_test)

In [ ]:
draw_mapes_hist(y_test, column_means)

In [ ]:
total_r2 = np.array([
    lasso_r2,
    ridge_r2,
    xgb_r2,
    rf_r2,
    et_r2,
    svr_r2,
    gbdt_r2
])

total_ci_width = np.array([
    lasso_ci_width,
    ridge_ci_width,
    xgb_ci_width,
    rf_ci_width,
    et_ci_width,
    svr_ci_width,
    gbdt_ci_width
])
total_stability = np.array([
    lasso_stability,
    ridge_stability,
    xgb_stability,
    rf_stability,
    et_stability,
    svr_stability,
    gbdt_stability
])

print(y_test)
final_r2 = r2_score(y_test, column_means)
final_ci_width = np.mean(total_ci_width)
final_stability = np.mean(total_stability) / 100
print("集成模型的R2：", final_r2)
print("集成模型的80%置信区间宽度：", final_ci_width)
print("集成模型的稳定性：",final_stability)

In [ ]:
total_importance = np.array([
    xgb_importance,
    rf_importance,
    et_importance,
    gbdt_importance
])
final_importance = np.mean(total_importance, axis=0)


importance_label = X_train.columns.values

paired_importance = list(zip(final_importance,importance_label))

paired_sorted_importance = sorted(paired_importance)
sorted_importance, sorted_label = zip(*paired_sorted_importance)

plt.figure(figsize=(10,20))
plt.barh(range(len(sorted_importance)),sorted_importance,tick_label=sorted_label)
plt.xlabel('Feature importance')
plt.title('Final feature importance')
plt.savefig('feature_importance.png')
plt.show()
#把该图保存到当前文件夹

In [ ]:
import pickle
with open('svr.pkl','wb') as file: pickle.dump(svr_model, file)
with open('gbdt.pkl','wb') as file: pickle.dump(gbdt_model, file)
with open('et.pkl','wb') as file: pickle.dump(et_model, file)
with open('rf.pkl','wb') as file: pickle.dump(rf_model, file)
with open('xgb.pkl','wb') as file: pickle.dump(xgb_model, file)

# V19.6 动态原生价格服务模型导出（Hotfix）

本单元不再假定必须存在七个模型。默认扫描Notebook当前工作目录中实际保存的模型文件，只把检查到的模型写入 `price_native_bundle.pkl`。服务运行时也只调用模型包中列出的成员。

默认识别：`lasso.pkl`、`ridge.pkl`、`xgb.pkl`、`rf.pkl`、`et.pkl`、`svr.pkl`、`gbdt.pkl`。例如前一单元只保存了5个模型，最终模型包就只包含这5个。未保存的Lasso/Ridge即使仍存在于Notebook内存中，也不会被自动加入。

权重处理：如果Notebook中的 `weights` 仍是七模型权重，导出器会按模型名称提取已保存模型对应的权重并重新归一化；如果权重数量正好等于已保存模型数量，则按已保存模型顺序使用；也可以显式传入 `{模型名: 权重}` 字典。

模型文件只能来自可信来源。标准库 `pickle` 不要求安装joblib，但运行服务仍需安装实际保存模型对应的依赖。例如没有保存XGBoost模型时，服务不需要XGBoost；保存了 `xgb.pkl` 时，就必须提供兼容的XGBoost版本。


In [ ]:
from pathlib import Path
import sys

# 修改为V19.6推荐系统根目录。
V19_6_PROJECT_ROOT = Path(r"D:\\IndustrialProtocolDemo_V19_6")
if str(V19_6_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(V19_6_PROJECT_ROOT))

from services.price_service.export_native_price_bundle import export_from_notebook

# 模型文件所在目录。原Notebook前一单元把svr.pkl、gbdt.pkl等保存在当前工作目录。
MODEL_DIRECTORY = Path.cwd()

# 使用自定义文件名时，在这里明确映射；使用默认文件名时保持None。
# 例如：{"svr": "my_svr.pkl", "random_forest": "final_rf.pkl"}
MODEL_FILE_MAP = None

# 把训练列映射为推荐系统/DataMaster使用的稳定字段编号。
FIELD_METADATA = {
    # "额定推力": {"field_name": "rated_thrust_n", "field_label": "额定推力", "unit": "N", "dtype": "number", "required": True, "missing_policy": "reject"},
    # "采购批量": {"field_name": "purchase_quantity", "field_label": "采购批量", "unit": "件", "dtype": "integer", "required": False, "missing_policy": "training_mean"},
}

# 原价格单位若为“元”，填写10000；已经是“万元”，填写1。
TARGET_DIVISOR_TO_WAN = 10000

bundle = export_from_notebook(
    globals(),
    output=V19_6_PROJECT_ROOT / "services" / "price_service" / "model" / "price_native_bundle.pkl",
    product_code="请填写与DataMaster一致的成品代号",
    product_name="请填写成品名称",
    model_version="price-native-dynamic-20260730",
    target_divisor_to_wan=TARGET_DIVISOR_TO_WAN,
    field_metadata=FIELD_METADATA,
    model_source="saved_files",
    saved_model_dir=MODEL_DIRECTORY,
    saved_model_files=MODEL_FILE_MAP,
)

print("已导出原生价格服务模型：", V19_6_PROJECT_ROOT / "services" / "price_service" / "model" / "price_native_bundle.pkl")
print("实际保存模型数量：", bundle["export_notes"]["included_model_count"])
print("实际保存模型：", bundle["export_notes"]["included_models"])
print("权重来源：", bundle["export_notes"]["weight_source"])
print("集成配置：", bundle["ensemble"])
print("所需依赖模块：", bundle["required_modules"])
